# Romania 2001: seven-county descriptive placement pilot

This notebook now includes **Alba (AB), Caraș-Severin (CS), Galați (GL), Tulcea (TL), Arad (AR), Sibiu (SB), and Bucharest–Ilfov (B)**. Its original filename is retained so your existing link still works. Change the visible `COUNTIES` tuple in Cell 1 to choose a subset. Other saved research settings are preserved. Bucharest uses the separately audited per-school result pages to fill its older county-report gaps. One duplicate result is explicitly reconciled to an independently recovered placement report. Existing displayed figures are from earlier runs until you rerun Cell 3.

We compare actual selective-program placements across gymnasium peer strength, including fixed own-examination-score bands corresponding to the questions in mosaic panels 7 and 8. Individual-county panels and a combined panel are saved for both the HERO and score-band views. These are descriptive associations; applicant preferences and causal assignment are not observed.

**Run order:** Cell 1 settings → Cell 2 county size and movement → Cell 3 researcher-controlled analysis → Cell 4 display a figure. Choose the `sports_net` kernel. No cell contacts Wayback. Raw student records stay in the private Desktop cache; each run writes only aggregate tables and figures into a new output folder.

The [original decision log](../docs/decisions/EDUCATION_20261002_Romania_four_county_pilot_design_decisions.md) preserves our definitions. The [Bucharest source gate](../outputs/romania_2001_bucharest_expansion_20261005/B_offline_source_gate.md) documents the recovered outcome source. The [Arad/Sibiu program reconciliation](../outputs/romania_2001_county_expansion_20261004/AR_SB_program_outcome_gate.md) records resolution of their formerly uncertain top-program labels. Expanding the county scope does not change the program-ranking rule or your score-band settings.

In [ ]:
# CELL 1 — VISIBLE RESEARCH SETTINGS
# This cell locates the repo from Cursor's current working directory. It imports
# the companion Python code, but importing that file DOES NOT fetch or analyze.
from pathlib import Path
import importlib
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

def find_analysis_root():
    for parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        candidate = parent / "3-Master_Plan/VECTOR_work/new_VECTOR_work/education_analysis"
        if candidate.is_dir():
            return candidate
        if parent.name == "education_analysis" and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("Open this notebook from inside the dissertation repository.")

ANALYSIS_ROOT = find_analysis_root()
CODE_DIR = ANALYSIS_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
# Reload the shared source readers too: the new county parameter and recovered
# program identities must be available even in an already-running kernel.
import EDUCATION_20261002_romania_full_gymnasium_offline_reconciliation as source_reader
import EDUCATION_20261003_romania_four_county_program_identity_gate as identity_reader
importlib.reload(source_reader)
importlib.reload(identity_reader)
import EDUCATION_20261004_romania_four_county_pilot as pilot
importlib.reload(pilot)  # Picks up a corrected .py file without restarting Cursor.

# Counties to include in this run. Change this tuple to compare a subset.
# AB Alba; CS Caraș-Severin; GL Galați; TL Tulcea; AR Arad; SB Sibiu;
# B Bucharest–Ilfov. B uses the separately audited school-result outcome pages.
COUNTIES = ("AB", "CS", "GL", "TL", "AR", "SB", "B")

# Primary definition agreed October 2–4: one highest cutoff tier, full programs,
# six original subjects, vocational destinations included. Tier 2 is a planned
# cumulative comparison, not a replacement chosen after looking at the answer.
TOP_TIERS = (1, 2)
REQUIRE_FULL_PROGRAMS = True
COMBINE_PROGRAM_CATEGORIES = False
INCLUDE_VOCATIONAL_IN_PRIMARY = True

# Separate, named comparisons. They do not change the primary definition.
COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS = True
COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS = True
COMPARE_POOLED_SCORE_BANDS = True

# Three percentile boundaries, ordered HIGH to LOW. Default (90, 75, 50)
# makes the red band top 10%, orange excellence band 75th–90th, gray
# 50th–75th, and blue below 50th. For example, (95, 80, 50) makes
# orange 80th–95th and red top 5%. These are OWN exam-score percentiles.
# Re-run Cell 1 and then Cell 3 after changing this tuple.
AI_PERCENTILE_CUTS = (90, 75, 50)

# Plot resolution only. These settings do not alter placement classifications.
PEER_STRENGTH_BINS = 8
HERO_BINS = 16

settings = pilot.Settings(
    counties=COUNTIES,
    top_tiers=TOP_TIERS,
    require_full_programs=REQUIRE_FULL_PROGRAMS,
    combine_program_categories=COMBINE_PROGRAM_CATEGORIES,
    include_vocational_in_primary=INCLUDE_VOCATIONAL_IN_PRIMARY,
    compare_excluding_ambiguous=COMPARE_EXCLUDING_AMBIGUOUS_EXACT_PROGRAMS,
    compare_excluding_vocational=COMPARE_EXCLUDING_VOCATIONAL_DESTINATIONS,
    compare_pooled_score_bands=COMPARE_POOLED_SCORE_BANDS,
    ai_percentile_cuts=AI_PERCENTILE_CUTS,
    peer_strength_bins=PEER_STRENGTH_BINS,
    hero_bins=HERO_BINS,
)
settings.validate()
print("Analysis code:", pilot.__file__)
print("Counties:", ", ".join(COUNTIES))
print("Top cutoff tiers:", TOP_TIERS, "| Full programs only:", REQUIRE_FULL_PROGRAMS)
print("Combine program categories:", COMBINE_PROGRAM_CATEGORIES)
print("Include vocational destinations:", INCLUDE_VOCATIONAL_IN_PRIMARY)
print("Own-score percentile cuts (top, excellence start, middle):", AI_PERCENTILE_CUTS)
print("No analysis or network request has run yet.")

## Cell 2: county size, incoming students, and outgoing students

This **offline source check** reads saved reports, verifies their checkpoints, and shows Alex's proposed county comparison before any plots are made.

- **Participating gymnasiums:** schools with at least one observed admission-round applicant. Schools listed in the directory can have zero applicants.
- **Outgoing:** an applicant from this county's gymnasium placed in another county. The percentage divides by all observed applicants from this county's gymnasiums.
- **Incoming:** a student from another county's gymnasium placed here. The percentage divides by all placements in this destination county.
- **Unresolved:** the saved evidence does not uniquely establish the outcome or origin. These cases are not assumed to be stayers.

Incoming and outgoing percentages use different denominators. They describe school locations, not family moves. Origin county is supported by recovered gymnasium pages or the printed origin-school county in the applicant report; names and admission scores provide provisional links because the original personal identifier is masked.

The county and program choices remain unchanged by this comparison. Cross-county placements remain part of their original gymnasium's peer calculation but outside the primary within-county outcome denominator.

In [ ]:
# CELL 2 — OFFLINE COUNTY SIZE AND MOVEMENT CHECK
# This reads private source records in memory; it prints only county aggregates.
# It does not request webpages, calculate HERO rates, or create output files.
county_overview = pilot.source_preview(settings)

display(county_overview[[
    "county_name", "county", "directory_gymnasiums", "participating_gymnasiums",
    "gymnasiums_with_two_or_more_applicants", "origin_applicants",
]].rename(columns={
    "county_name": "County", "county": "Code",
    "directory_gymnasiums": "Gymnasiums in directory",
    "participating_gymnasiums": "Gymnasiums with applicants",
    "gymnasiums_with_two_or_more_applicants": "Gymnasiums with 2+ applicants",
    "origin_applicants": "Applicants from these gymnasiums",
}))
display(county_overview[[
    "county", "outgoing_students", "outgoing_pct_of_origin_applicants",
    "incoming_students", "incoming_pct_of_destination_placements",
    "origin_outcome_unresolved", "destination_origin_unresolved",
]].rename(columns={
    "county": "Code", "outgoing_students": "Placed elsewhere",
    "outgoing_pct_of_origin_applicants": "Outgoing % of origin applicants",
    "incoming_students": "Placed here from elsewhere",
    "incoming_pct_of_destination_placements": "Incoming % of placements here",
    "origin_outcome_unresolved": "Origin applicants: outcome unresolved",
    "destination_origin_unresolved": "Placements here: origin/match unresolved",
}).round(2))
print("The two percentages have different denominators; do not subtract them.")
print("Cell 3 saves this comparison with the plots in county_size_and_movement.csv.")


## Cell 3: you start the analysis

Set `RUN_ANALYSIS = True` and run the next cell when you are ready. The companion will print four progress stages: verifying saved pages; setting exam-score bands; calculating county outcomes and plot cells; saving tables and figures. It makes **no Wayback requests**. A new timestamped folder preserves every prior run.

The program rule is applied separately in each county. The red and orange bands describe **own national-exam performance within the originating county**, using all recovered admission-round applicants before outcome exclusions. Cell 1 sets their percentile boundaries with `AI_PERCENTILE_CUTS`; the default gives red = top 10% and orange = 75th–90th. A separately saved comparison uses pooled selected-county score bands. The horizontal axis uses the **original exam-score scale** for the other observed applicants in the gymnasium.

In [ ]:
# CELL 3 — RESEARCHER-CONTROLLED ANALYSIS
# Leave this False to inspect settings and source counts without calculating
# placement rates or writing plots. Charles changes it to True in Cursor.
RUN_ANALYSIS = False

if not RUN_ANALYSIS:
    print("Analysis did not run. Change RUN_ANALYSIS to True, then rerun this cell.")
else:
    result = pilot.run_pilot(settings, run=True)
    run_folder, summary = result
    print("\nSaved this run in:", run_folder)
    print("Read first:", run_folder / "report.md")
    display(summary[[
        "county", "tier", "source_applicants", "primary_denominator",
        "primary_successes", "primary_observed_success_fraction",
        "qualifying_programs", "qualifying_program_places",
    ]].rename(columns={
        "county": "County", "tier": "Top tiers",
        "source_applicants": "Source applicants",
        "primary_denominator": "Outcome denominator",
        "primary_successes": "Selected",
        "primary_observed_success_fraction": "Observed fraction",
        "qualifying_programs": "Winning programs",
        "qualifying_program_places": "Program places",
    }))

## Cell 4: read the report and choose a figure

After Cell 3 runs, choose `TIER_TO_VIEW` and `FIGURE_TO_VIEW`: `"hero"`, `"combined_hero"`, `"county_bands"`, or `"pooled_bands"`.

With all six counties selected, each score-band figure has **seven data panels: six individual counties and one combined panel**. The HERO figure has seven rows, each with equal-width and quantile bins; `"combined_hero"` shows the combined row alone. Red and orange use your own-exam percentile settings in Cell 1. The county-band view defines excellence separately in each county; the pooled-band comparison uses one set of score boundaries across the selected counties.

Point labels show successful placements/applicants and distinct gymnasiums. Combined labels also show counties represented. Larger counties contribute more applicants to the combined curve; it is not a county-adjusted effect. All bin counts and the county size/movement comparison are saved as CSV files.

In [ ]:
# CELL 4 — DISPLAY SAVED OUTPUTS ONLY; NO RECOMPUTATION
TIER_TO_VIEW = 1
FIGURE_TO_VIEW = "county_bands"  # "hero", "combined_hero", "county_bands", or "pooled_bands"

if "run_folder" not in globals():
    print("Cell 3 has not produced a run folder in this kernel.")
else:
    display(Markdown((run_folder / "report.md").read_text(encoding="utf-8")))
    filename = {
        "hero": f"hero_top{TIER_TO_VIEW}.png",
        "combined_hero": f"hero_combined_top{TIER_TO_VIEW}.png",
        "county_bands": f"conditional_county_bands_top{TIER_TO_VIEW}.png",
        "pooled_bands": f"conditional_pooled_bands_top{TIER_TO_VIEW}.png",
    }[FIGURE_TO_VIEW]
    figure_path = run_folder / filename
    if not figure_path.exists():
        print("This figure was not requested by the current settings:", figure_path)
    else:
        print("Displaying:", figure_path)
        display(Image(filename=str(figure_path)))

## How to read this pilot

A curve compares **actual placements** among applicants who spent time in different originating gymnasiums and had similar measured exam performance. It does not tell us which programs they applied for, whether their classmates caused their placement, or whether an observed downturn is congestion. The report must keep those limits visible beside any promising pattern.

The notebook intentionally does not delete unresolved source items or classify the one missing Galați outcome. Six-county outputs are saved under `education_analysis/outputs/romania_2001_6_county_descriptive_pilot/`, one timestamped folder per run. Original four-county runs retain their previous output location. The raw Ministry pages remain outside the repository.

## Cell 5: CROSS-COUNTY OUTCOME LEDGER — source check

This section keeps the Cell 1 county list, top-one/top-two program rules, subject categories, full-program requirement, and own-score bands. It adds a second definition of success: a top program in the actual destination county. A student placed outside the origin county counts as zero for a *home-county* top program; their *anywhere-observed* outcome is included only when the destination program can be matched and ranked. Both plotted lines use exactly the same people.

Run the next code cell to inspect source counts only. It makes no web requests and saves no plots. It should report 576 outward-bound placements, with 548 currently rankable and 28 unresolved. Unknown program ranks stay unresolved.


In [ ]:
# CELL 5 — SOURCE-ONLY PREVIEW OF HOME / ANYWHERE OUTCOMES
# Reload the companion code after any agent edit. This cell checks saved pages
# and prints name-free counts. It does not run the descriptive comparison.
import EDUCATION_20261007_romania_cross_county_outcome_ledger as cross_county
importlib.reload(cross_county)
cross_county_audit = cross_county.audit_sources(settings)
print("Rankable outward-bound placements:", cross_county_audit.get("away_placed", 0))
print("Unresolved destination ranks:",
      cross_county_audit.get("program_rank_mixed", 0)
      + cross_county_audit.get("destination_catalog_absent", 0))


## Cell 6: you start the paired comparison

Set `RUN_CROSS_COUNTY_COMPARISON = True` when you want the new figures. The row-level outcome ledger is saved only on your Desktop in `VECTOR_temp`; the dissertation folder receives name-free aggregate tables, source counts, a report, and paired plots. The paired figures show local versus anywhere-observed top-program placement within the same own-examination-score bands. A movement table shows whether outward-bound applicants have different examination scores. This is descriptive; the archive does not reveal students preferred programs or why they applied across county lines.

The figures plot only cells with at least 20 applicants. Smaller cells remain in the aggregate CSV. This display threshold does not change who enters the paired denominator.


In [ ]:
# CELL 6 — RESEARCHER-CONTROLLED PAIRED PLOTS
# Cell 1 must have run first. Leave False while inspecting sources.
RUN_CROSS_COUNTY_COMPARISON = False
SAVE_PRIVATE_OUTCOME_LEDGER = True

if not RUN_CROSS_COUNTY_COMPARISON:
    print("Comparison did not run. Change RUN_CROSS_COUNTY_COMPARISON to True, then rerun this cell.")
else:
    comparison_folder, comparison_audit, movement_table = cross_county.run_comparison(
        settings, run=True, save_private_ledger=SAVE_PRIVATE_OUTCOME_LEDGER
    )
    print("Read the plain-language report:", comparison_folder / "report.md")
    display(comparison_audit)
    display(movement_table)
    # Saved PNGs display reliably in Cursor even when Matplotlib only prints
    # a Figure label instead of rendering the image in notebook output.
    display(Image(filename=str(comparison_folder / "paired_excellence_top1.png")))
    display(Image(filename=str(comparison_folder / "paired_pooled_bands_top1.png")))
    display(Image(filename=str(comparison_folder / "movement_exam_by_status.png")))
    display(Image(filename=str(comparison_folder / "multistate_excellence.png")))
    print("Top-two paired figures and all aggregate CSVs are saved in the same folder.")
